<a href="https://colab.research.google.com/github/ZorahSai/Projeto-WebAgriculture-/blob/main/Coleta%20de%20Dados/%20dados_ripe_atlas_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coleta de dados — RIPE Atlas (pares origem → destino, múltiplos destinos)

Este notebook gera o CSV do projeto **Preditor de Falha e Risco em Dispositivos de Rede**.

**Ideia central:** cada linha do dataset pertence a um **par** (origem → destino): uma Anchor de origem (`prb_id`) medindo uma Anchor de destino com `ping` contínuo. O notebook:

1. parte de uma lista de **20 Anchors de destino** (7 países);
2. para cada destino, encontra uma medição pública **contínua** que o mede;
3. entre as origens dessa medição, mantém só as da **mesma região (continente)** do destino e sorteia de **3 a 5**, em **países variados**;
4. coleta o histórico, valida, transforma para o Contrato de Dados e classifica `OK / RISCO / FALHA` em relação à linha de base de cada par.

> Este notebook é independente do passo a passo da professora: a etapa de descoberta (Anchor → medição → origens) foi incorporada aqui como funções, sem depender de variáveis de outro notebook.

## 1. Imports e pasta de destino (`raw`)

Funciona no Google Colab (salva no Drive) e localmente, no VS Code (salva em `./ripe_atlas/raw`).

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import requests
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PROJETO = Path("/content/drive/MyDrive/ripe_atlas")
else:
    PASTA_PROJETO = Path("./ripe_atlas")

PASTA_RAW = PASTA_PROJETO / "raw"
PASTA_RAW.mkdir(parents=True, exist_ok=True)

print("Arquivos raw serão salvos em:", PASTA_RAW)

Mounted at /content/drive
Arquivos raw serão salvos em: /content/drive/MyDrive/ripe_atlas/raw


## 2. Parâmetros e decisões da equipe

| Decisão | Escolha da equipe | Justificativa |
|---|---|---|
| Destinos | 20 Anchors em 7 países (BR 5, US 3, CO 3, DE 3, JP 2, NZ 2, FI 2) | Atende à devolutiva: vários destinos em regiões diferentes, em vez de uma única Anchor |
| Tipo de medição | `ping` IPv4, pública e **contínua** (`is_oneoff=false`) | Gera série temporal real por par; medições one-off não sustentam histórico |
| Origens por destino | de 3 a 5 | Quantidade controlada por destino |
| Região das origens | mesma região (continente) do destino | Ex.: Brasil com origens na América do Sul; Japão com origens na Ásia |
| Diversidade | no máximo 2 origens por país; sorteio com semente fixa (42) | Evita concentrar as origens em um único país e torna a seleção reproduzível |
| Tipo de origem | apenas Anchors públicas (`anchors_only`, `public_only`) | Origens estáveis e conhecidas |
| Janela de coleta | 168 horas (7 dias), terminando no mesmo instante para todos os destinos | Mais exemplos de RISCO/FALHA e ciclo semanal; janela comum permite comparar destinos |
| Formato de saída | CSV | Leve, legível e compatível com pandas/Excel |
| Rótulo `status_real` | relativo à linha de base de cada par (média ± desvio-padrão da latência) | O RTT normal depende da distância; um limiar fixo rotularia como falha pares distantes que estão saudáveis |

Os parâmetros ficam todos nesta célula, antes de qualquer uso.

In [ ]:
BASE_URL = "https://atlas.ripe.net/api/v2"

# Anchors de destino escolhidas pela equipe (o país vem do prefixo do FQDN)
DESTINOS_FQDN = [
    # Brasil
    "br-for-as37468.anchors.atlas.ripe.net",
    "br-pgz-as53046.anchors.atlas.ripe.net",
    "br-poa-as19200.anchors.atlas.ripe.net",
    "br-sao-as12008.anchors.atlas.ripe.net",
    "br-fec-as264479.anchors.atlas.ripe.net",
    # Estados Unidos
    "us-lax-as30058.anchors.atlas.ripe.net",
    "us-mes-as401332.anchors.atlas.ripe.net",
    "us-mia-as199610.anchors.atlas.ripe.net",
    # Japão
    "jp-tyo-as63834.anchors.atlas.ripe.net",
    "jp-tch-as63806.anchors.atlas.ripe.net",
    # Nova Zelândia
    "nz-npl-as132254.anchors.atlas.ripe.net",
    "nz-akl-as138398.anchors.atlas.ripe.net",
    # Colômbia
    "co-ptx-as269907.anchors.atlas.ripe.net",
    "co-bog-as273103.anchors.atlas.ripe.net",
    "co-sdq-as271927.anchors.atlas.ripe.net",
    # Alemanha
    "de-muc-as15456.anchors.atlas.ripe.net",
    "de-mdt-as56368.anchors.atlas.ripe.net",
    "de-mai-as2857.anchors.atlas.ripe.net",
    # Finlândia
    "fi-tmp-as719.anchors.atlas.ripe.net",
    "fi-sal-as39699.anchors.atlas.ripe.net",
]

# Seleção das origens
MIN_PROBES = 3               # mínimo de origens por destino
MAX_PROBES = 5               # máximo de origens por destino
MAX_POR_PAIS = 2             # no máximo 2 origens do mesmo país por destino
MAX_MEDICOES_TESTADAS = 5    # quantas medições candidatas testar por destino
SEMENTE = 42                 # sorteio reprodutível

# Coleta
JANELA_HORAS = 168           # 7 dias
FIM_COLETA_UTC = None        # None = agora; ou "2026-10-08T13:53:39+00:00" para repetir exatamente a mesma janela
TIMEOUT_REQUISICAO = 60
FORMATO_TABELAR = "csv"

# Regiões (continentes) usadas para restringir as origens
REGIOES = {
    "América do Sul": ["AR", "BO", "BR", "CL", "CO", "EC", "GY", "PY", "PE", "SR", "UY", "VE"],
    "América do Norte": ["US", "CA", "MX", "GT", "CR", "PA", "HN", "SV", "NI", "BZ", "CU", "DO", "JM", "PR", "BS"],
    "Europa": ["AL", "AT", "BA", "BE", "BG", "BY", "CH", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", "FR",
               "GB", "GR", "HR", "HU", "IE", "IS", "IT", "LI", "LT", "LU", "LV", "MD", "ME", "MK", "MT",
               "NL", "NO", "PL", "PT", "RO", "RS", "SE", "SI", "SK", "UA", "XK"],
    "Ásia": ["JP", "CN", "KR", "IN", "SG", "TH", "ID", "MY", "VN", "PH", "HK", "TW", "BD", "PK", "LK",
             "NP", "KH", "LA", "MM", "MN", "AE", "IL", "SA", "QA", "KW", "IR", "IQ", "JO", "LB", "OM",
             "BH", "AM", "AZ", "GE"],
    "Oceania": ["AU", "NZ", "FJ", "NC", "PG", "PF", "WS", "TO", "VU", "GU"],
    "África": ["ZA", "EG", "NG", "KE", "MA", "TN", "DZ", "GH", "TZ", "UG", "SN", "MU", "RW", "ZM", "ZW",
               "MZ", "CI", "CM"],
}
REGIAO_POR_PAIS = {pais: regiao for regiao, paises in REGIOES.items() for pais in paises}

CACHE_PAIS_PROBE = {}        # prb_id -> country_code (evita repetir consultas)

# Verificações dos parâmetros (falham cedo, antes de chamar a API)
assert 1 <= MIN_PROBES <= MAX_PROBES, "MIN_PROBES deve ser >= 1 e <= MAX_PROBES"
assert MAX_POR_PAIS >= 1
assert FORMATO_TABELAR in {"csv", "parquet"}
assert len(DESTINOS_FQDN) == len(set(DESTINOS_FQDN)), "Há FQDNs repetidos em DESTINOS_FQDN"
assert sum(len(v) for v in REGIOES.values()) == len(REGIAO_POR_PAIS), "Há país repetido em mais de uma região"
if FIM_COLETA_UTC is not None:
    assert datetime.fromisoformat(FIM_COLETA_UTC).tzinfo is not None, "FIM_COLETA_UTC precisa de fuso (ex.: +00:00)"

paises_sem_regiao = sorted({f.split("-")[0].upper() for f in DESTINOS_FQDN} - set(REGIAO_POR_PAIS))
assert not paises_sem_regiao, f"Países de destino sem região mapeada: {paises_sem_regiao}"

print("Destinos:", len(DESTINOS_FQDN))
print(pd.Series([REGIAO_POR_PAIS[f.split("-")[0].upper()] for f in DESTINOS_FQDN]).value_counts().to_string())

Destinos: 20
América do Sul      8
Europa              5
América do Norte    3
Ásia                2
Oceania             2


## 3. Funções de acesso à API

Aqui está a etapa de descoberta do passo a passo da professora, generalizada para vários destinos:

- `requisitar`: GET com novas tentativas para erros temporários (timeout, 429, 5xx).
- `obter_metadados_medicao`: lê `is_oneoff`, `interval`, `packets` da medição.
- `buscar_medicoes_continuas`: acha medições públicas de ping **contínuas** que têm a Anchor como alvo.
- `obter_paises_probes`: descobre o país de cada origem (em lotes de 50, com cache).
- `candidatos_mesma_regiao`: lista as Anchors que mediram o destino nos últimos 30 minutos e estão na mesma região.
- `selecionar_diversificado`: escolhe de 3 a 5 origens em rodízio por país (uma de cada país primeiro, depois uma segunda de cada país).

In [ ]:
def requisitar(url, params=None, timeout_requisicao=60, tentativas=3):
    """GET com novas tentativas para erros temporários (timeout, 429 e 5xx)."""
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, params=params, timeout=timeout_requisicao)
            if resposta.status_code in (429, 500, 502, 503, 504) and tentativa < tentativas:
                time.sleep(2 ** tentativa)
                continue
            resposta.raise_for_status()
            return resposta
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if tentativa == tentativas:
                raise
            time.sleep(2 ** tentativa)


def obter_metadados_medicao(measurement_id, timeout_requisicao):
    info = requisitar(f"{BASE_URL}/measurements/{measurement_id}/", None, timeout_requisicao).json()
    return {
        "is_oneoff": info.get("is_oneoff"),
        "interval": info.get("interval"),
        "packets": info.get("packets"),
        "start_time": info.get("start_time"),
        "stop_time": info.get("stop_time"),
    }


def buscar_medicoes_continuas(fqdn, timeout_requisicao):
    params = {"target": fqdn, "type": "ping", "af": 4, "is_public": "true", "is_oneoff": "false"}
    return requisitar(f"{BASE_URL}/measurements/", params, timeout_requisicao).json()["results"]


def obter_paises_probes(probe_ids, timeout_requisicao):
    faltantes = [p for p in probe_ids if p not in CACHE_PAIS_PROBE]
    for i in range(0, len(faltantes), 50):
        lote = faltantes[i:i + 50]
        try:
            params = {"id__in": ",".join(str(p) for p in lote), "page_size": 100}
            for probe in requisitar(f"{BASE_URL}/probes/", params, timeout_requisicao).json()["results"]:
                CACHE_PAIS_PROBE[probe["id"]] = probe.get("country_code")
        except requests.exceptions.HTTPError:
            for prb_id in lote:  # alternativa: uma consulta por probe
                resposta = requests.get(f"{BASE_URL}/probes/{prb_id}/", timeout=timeout_requisicao)
                if resposta.status_code == 200:
                    CACHE_PAIS_PROBE[prb_id] = resposta.json().get("country_code")
    return {p: CACHE_PAIS_PROBE.get(p) for p in probe_ids}


def candidatos_mesma_regiao(measurement_id, regiao_destino, timeout_requisicao):
    fim = datetime.now(timezone.utc)
    inicio = fim - timedelta(minutes=30)
    params = {
        "start": int(inicio.timestamp()), "stop": int(fim.timestamp()),
        "anchors_only": "true", "public_only": "true", "format": "json",
    }
    resultados = requisitar(
        f"{BASE_URL}/measurements/{measurement_id}/results/", params, timeout_requisicao
    ).json()
    participantes = sorted({r["prb_id"] for r in resultados if "prb_id" in r})
    paises = obter_paises_probes(participantes, timeout_requisicao)
    return [p for p in participantes if REGIAO_POR_PAIS.get(paises[p]) == regiao_destino]


def selecionar_diversificado(candidatos, min_probes, max_probes, max_por_pais, rng):
    por_pais = {}
    for prb_id in candidatos:
        por_pais.setdefault(CACHE_PAIS_PROBE[prb_id], []).append(prb_id)
    for lista in por_pais.values():
        rng.shuffle(lista)

    paises = sorted(por_pais)
    rng.shuffle(paises)

    escolhidos = []
    for rodada in range(max_por_pais):          # rodada 0: 1 por país; rodada 1: 2º de cada país
        for pais in paises:
            if len(escolhidos) >= max_probes:
                break
            if rodada < len(por_pais[pais]):
                escolhidos.append(por_pais[pais][rodada])

    # Se a região tem poucos países e o mínimo não foi atingido, completa relaxando o limite por país
    if len(escolhidos) < min_probes:
        restantes = [p for p in candidatos if p not in escolhidos]
        rng.shuffle(restantes)
        escolhidos += restantes[: min_probes - len(escolhidos)]

    return sorted(escolhidos)

## 4. Seleção dos pares (origens de cada destino)

Para cada Anchor de destino: acha uma medição contínua que a mede, lista as origens da mesma região e sorteia de 3 a 5 em países variados. Cada linha `[OK]` mostra os países sorteados, o que permite conferir a diversidade antes da coleta. Destinos sem medição contínua adequada aparecem como `[PULADO]` e ficam registrados em `pares_selecionados.json`.

In [ ]:
configuracao_destinos = []
destinos_ignorados = []

for fqdn in DESTINOS_FQDN:
    pais_destino = fqdn.split("-")[0].upper()
    regiao = REGIAO_POR_PAIS[pais_destino]

    try:
        escolhida = None
        for medicao in buscar_medicoes_continuas(fqdn, TIMEOUT_REQUISICAO)[:MAX_MEDICOES_TESTADAS]:
            candidatos = candidatos_mesma_regiao(medicao["id"], regiao, TIMEOUT_REQUISICAO)
            if len(candidatos) >= MIN_PROBES:
                escolhida = (medicao["id"], candidatos)
                break
    except requests.exceptions.RequestException as erro:
        print(f"[ERRO]   {fqdn}: {erro}")
        destinos_ignorados.append([fqdn, f"erro de API: {erro}"])
        continue

    if escolhida is None:
        print(f"[PULADO] {fqdn}: nenhuma medição contínua com {MIN_PROBES}+ origens em {regiao}.")
        destinos_ignorados.append([fqdn, f"sem medição contínua com {MIN_PROBES}+ origens em {regiao}"])
        continue

    measurement_id, candidatos = escolhida
    rng = random.Random(f"{SEMENTE}-{fqdn}")
    sorteados = selecionar_diversificado(candidatos, MIN_PROBES, MAX_PROBES, MAX_POR_PAIS, rng)
    paises_sorteados = [CACHE_PAIS_PROBE[p] for p in sorteados]

    print(f"[OK]     {fqdn} | {regiao} | msm {measurement_id} | "
          f"{len(candidatos)} candidatas -> {len(sorteados)} origens {paises_sorteados}")
    configuracao_destinos.append({
        "anchor_fqdn": fqdn, "regiao": regiao, "measurement_id": measurement_id,
        "prb_ids": sorteados, "paises_origem": paises_sorteados,
    })
    time.sleep(0.3)

sem_regiao = sorted({c for c in CACHE_PAIS_PROBE.values() if c and c not in REGIAO_POR_PAIS})
if sem_regiao:
    print("\nPaíses de origem sem região mapeada (ignorados):", sem_regiao)
if destinos_ignorados:
    print(f"\nATENÇÃO: {len(destinos_ignorados)} destino(s) ficaram fora do dataset:")
    for fqdn, motivo in destinos_ignorados:
        print(" -", fqdn, "->", motivo)

assert configuracao_destinos, "Nenhum destino foi configurado — revise os parâmetros."

with (PASTA_RAW / "pares_selecionados.json").open("w", encoding="utf-8") as arquivo:
    json.dump({
        "parametros": {
            "min_probes": MIN_PROBES, "max_probes": MAX_PROBES, "max_por_pais": MAX_POR_PAIS,
            "semente": SEMENTE, "janela_horas": JANELA_HORAS,
        },
        "destinos": configuracao_destinos,
        "ignorados": destinos_ignorados,
    }, arquivo, ensure_ascii=False, indent=2)

display(pd.DataFrame([{
    "destino": c["anchor_fqdn"], "regiao": c["regiao"], "measurement_id": c["measurement_id"],
    "origens": len(c["prb_ids"]), "paises_origem": ", ".join(c["paises_origem"]),
} for c in configuracao_destinos]))

[OK]     br-for-as37468.anchors.atlas.ripe.net | América do Sul | msm 57650135 | 52 candidatas -> 5 origens ['CO', 'VE', 'CL', 'UY', 'BO']
[OK]     br-pgz-as53046.anchors.atlas.ripe.net | América do Sul | msm 31555548 | 52 candidatas -> 5 origens ['BR', 'UY', 'AR', 'PE', 'CO']
[OK]     br-poa-as19200.anchors.atlas.ripe.net | América do Sul | msm 11688943 | 52 candidatas -> 5 origens ['BR', 'CO', 'VE', 'EC', 'AR']
[OK]     br-sao-as12008.anchors.atlas.ripe.net | América do Sul | msm 29650317 | 52 candidatas -> 5 origens ['BR', 'UY', 'PE', 'VE', 'CO']
[OK]     br-fec-as264479.anchors.atlas.ripe.net | América do Sul | msm 66196984 | 52 candidatas -> 5 origens ['EC', 'CL', 'VE', 'PE', 'CO']
[OK]     us-lax-as30058.anchors.atlas.ripe.net | América do Norte | msm 158773662 | 184 candidatas -> 5 origens ['MX', 'PA', 'GT', 'US', 'CR']
[OK]     us-mes-as401332.anchors.atlas.ripe.net | América do Norte | msm 80082935 | 179 candidatas -> 5 origens ['US', 'CA', 'PA', 'CR', 'SV']
[OK]     us-mia-as

,destino,regiao,measurement_id,origens,paises_origem
0,br-for-as37468.anchors.atlas.ripe.net,América do Sul,57650135,5,"CO, VE, CL, UY, BO"
1,br-pgz-as53046.anchors.atlas.ripe.net,América do Sul,31555548,5,"BR, UY, AR, PE, CO"
2,br-poa-as19200.anchors.atlas.ripe.net,América do Sul,11688943,5,"BR, CO, VE, EC, AR"
3,br-sao-as12008.anchors.atlas.ripe.net,América do Sul,29650317,5,"BR, UY, PE, VE, CO"
4,br-fec-as264479.anchors.atlas.ripe.net,América do Sul,66196984,5,"EC, CL, VE, PE, CO"
5,us-lax-as30058.anchors.atlas.ripe.net,América do Norte,158773662,5,"MX, PA, GT, US, CR"
6,us-mes-as401332.anchors.atlas.ripe.net,América do Norte,80082935,5,"US, CA, PA, CR, SV"
7,us-mia-as199610.anchors.atlas.ripe.net,América do Norte,22523924,5,"CA, PA, HN, US, GT"
8,jp-tyo-as63834.anchors.atlas.ripe.net,Ásia,22371490,5,"TH, ID, AZ, NP, KR"
9,jp-tch-as63806.anchors.atlas.ripe.net,Ásia,84216561,5,"GE, PH, IQ, KR, IN"


## 5. Coleta, validação e transformação — por destino

Todos os destinos usam a **mesma janela** (`inicio_utc` → `fim_utc`), calculada uma única vez. Para cada destino, uma única chamada à API traz o histórico das 3 a 5 origens (`probe_ids=...`).

Para não carregar ~200 mil registros brutos na memória de uma vez, cada destino é processado e descartado em sequência: salva o bruto em `raw/por_destino/` (JSON original, tabela e metadados), calcula a validação e converte para o Contrato de Dados.

In [ ]:
def coletar_resultados_pares(measurement_id, prb_ids, destino_fqdn, inicio_utc, fim_utc,
                             timeout_requisicao, pasta_raw, formato_tabelar="csv"):
    metadados_medicao = obter_metadados_medicao(measurement_id, timeout_requisicao)
    assert metadados_medicao["is_oneoff"] is False, f"A medição {measurement_id} é one-off."

    url = f"{BASE_URL}/measurements/{measurement_id}/results/"
    parametros = {
        "start": int(inicio_utc.timestamp()), "stop": int(fim_utc.timestamp()),
        "probe_ids": ",".join(str(p) for p in prb_ids), "format": "json",
    }

    instante_requisicao_utc = datetime.now(timezone.utc)
    resposta = requisitar(url, parametros, timeout_requisicao)
    dados_json = resposta.json()
    if len(dados_json) == 0:
        raise RuntimeError(f"Nenhum registro retornado para {destino_fqdn}.")

    df_raw = pd.json_normalize(dados_json, sep="_")

    pasta_destino = pasta_raw / "por_destino"
    pasta_destino.mkdir(parents=True, exist_ok=True)
    nome_base = (f"ripe_atlas_{destino_fqdn.split('.')[0]}_m{measurement_id}_"
                 f"{instante_requisicao_utc.strftime('%Y%m%dT%H%M%SZ')}")

    caminho_json = pasta_destino / f"{nome_base}.json"
    with caminho_json.open("w", encoding="utf-8") as arquivo:
        json.dump(dados_json, arquivo, ensure_ascii=False)

    caminho_tabela = pasta_destino / f"{nome_base}.{formato_tabelar}"
    if formato_tabelar == "csv":
        df_raw.to_csv(caminho_tabela, index=False, encoding="utf-8")
    else:
        df_raw.to_parquet(caminho_tabela, index=False)

    metadados = {
        "destino_fqdn": destino_fqdn,
        "measurement_id": measurement_id,
        "is_oneoff": metadados_medicao["is_oneoff"],
        "interval": metadados_medicao["interval"],
        "packets": metadados_medicao["packets"],
        "pares_prb_ids": list(prb_ids),
        "url": url,
        "parametros": parametros,
        "inicio_utc": inicio_utc.isoformat(),
        "fim_utc": fim_utc.isoformat(),
        "coletado_em_utc": instante_requisicao_utc.isoformat(),
        "status_http": resposta.status_code,
        "quantidade_registros": len(dados_json),
        "quantidade_colunas_dataframe": len(df_raw.columns),
        "arquivo_json": caminho_json.name,
        "arquivo_tabelar": caminho_tabela.name,
    }
    with (pasta_destino / f"{nome_base}_metadata.json").open("w", encoding="utf-8") as arquivo:
        json.dump(metadados, arquivo, ensure_ascii=False, indent=2)

    return {"df_raw": df_raw, "dados_json": dados_json, "metadados": metadados}


def validar_pares(df_raw, dados_json):
    por_par = df_raw.groupby("prb_id").size()
    return {
        "pares": int(df_raw["prb_id"].nunique()),
        "registros": int(len(df_raw)),
        "registros_min_por_par": int(por_par.min()),
        "registros_max_por_par": int(por_par.max()),
        "duplicatas": int(df_raw.duplicated(subset=["prb_id", "timestamp"]).sum()),
        "timeout_total": int((df_raw["rcvd"] == 0).sum()),
        "pacotes_perdidos": sum(1 for r in dados_json for p in r.get("result", []) if "x" in p),
    }


def transformar_para_contrato(dados_json, destino_fqdn):
    registros = []
    for sonda in dados_json:
        enviados, recebidos = sonda["sent"], sonda["rcvd"]
        perda_pct = ((enviados - recebidos) / enviados) * 100 if enviados > 0 else 100
        rtts = [r["rtt"] for r in sonda.get("result", []) if "rtt" in r]
        if len(rtts) > 1:
            jitter = sum(abs(rtts[i] - rtts[i - 1]) for i in range(1, len(rtts))) / (len(rtts) - 1)
        else:
            jitter = 0.0

        registros.append({
            "timestamp": pd.to_datetime(sonda["timestamp"], unit="s", utc=True),
            "par_id": f"{sonda['prb_id']}->{destino_fqdn}",
            "prb_id": sonda["prb_id"],
            "ip": sonda["src_addr"],
            "latencia_ms": round(sonda.get("avg", 0), 2),
            "perda_pacotes_pct": round(perda_pct, 2),
            "jitter_ms": round(jitter, 2),
        })
    return pd.DataFrame(registros)

In [ ]:
# Janela comum a todos os destinos
fim_utc = datetime.fromisoformat(FIM_COLETA_UTC) if FIM_COLETA_UTC else datetime.now(timezone.utc)
inicio_utc = fim_utc - timedelta(hours=JANELA_HORAS)
print("Janela de coleta (UTC):", inicio_utc.isoformat(), "->", fim_utc.isoformat())

resultados_validacao = []
partes_contrato = []

for config in configuracao_destinos:
    fqdn = config["anchor_fqdn"]
    resultado = coletar_resultados_pares(
        config["measurement_id"], config["prb_ids"], fqdn, inicio_utc, fim_utc,
        TIMEOUT_REQUISICAO, PASTA_RAW, FORMATO_TABELAR,
    )

    validacao = validar_pares(resultado["df_raw"], resultado["dados_json"])
    validacao.update({
        "destino_fqdn": fqdn, "regiao_destino": config["regiao"],
        "pares_solicitados": len(config["prb_ids"]),
        "interval_s": resultado["metadados"]["interval"],
    })
    resultados_validacao.append(validacao)

    parte = transformar_para_contrato(resultado["dados_json"], fqdn)
    parte["destino_fqdn"] = fqdn
    parte["regiao_destino"] = config["regiao"]
    parte["pais_origem"] = parte["prb_id"].map(CACHE_PAIS_PROBE)
    partes_contrato.append(parte)

    print(f"{fqdn}: {validacao['registros']} registros | "
          f"{validacao['pares']}/{validacao['pares_solicitados']} pares com dados")

    del resultado
    time.sleep(0.3)

Janela de coleta (UTC): 2026-10-03T22:06:58.230119+00:00 -> 2026-10-10T22:06:58.230119+00:00
br-for-as37468.anchors.atlas.ripe.net: 12506 registros | 5/5 pares com dados
br-pgz-as53046.anchors.atlas.ripe.net: 12497 registros | 5/5 pares com dados
br-poa-as19200.anchors.atlas.ripe.net: 12540 registros | 5/5 pares com dados
br-sao-as12008.anchors.atlas.ripe.net: 11628 registros | 5/5 pares com dados
br-fec-as264479.anchors.atlas.ripe.net: 12549 registros | 5/5 pares com dados
us-lax-as30058.anchors.atlas.ripe.net: 12535 registros | 5/5 pares com dados
us-mes-as401332.anchors.atlas.ripe.net: 12532 registros | 5/5 pares com dados
us-mia-as199610.anchors.atlas.ripe.net: 12522 registros | 5/5 pares com dados
jp-tyo-as63834.anchors.atlas.ripe.net: 12527 registros | 5/5 pares com dados
jp-tch-as63806.anchors.atlas.ripe.net: 12554 registros | 5/5 pares com dados
nz-npl-as132254.anchors.atlas.ripe.net: 12545 registros | 5/5 pares com dados
nz-akl-as138398.anchors.atlas.ripe.net: 12558 registros 

## 6. Validação da coleta

Conferência de integridade por destino: pares com dados (comparado ao que foi solicitado), registros por par, duplicatas, timeouts totais (`rcvd == 0`) e pacotes individuais perdidos (`"x"` em `result`).

In [ ]:
df_validacao = pd.DataFrame(resultados_validacao)[[
    "regiao_destino", "destino_fqdn", "interval_s", "pares_solicitados", "pares", "registros",
    "registros_min_por_par", "registros_max_por_par", "duplicatas", "timeout_total", "pacotes_perdidos",
]]
display(df_validacao)

print("Destinos:", len(df_validacao), "| Pares:", int(df_validacao["pares"].sum()),
      "| Registros:", int(df_validacao["registros"].sum()))
print("Duplicatas:", int(df_validacao["duplicatas"].sum()),
      "| Timeouts totais:", int(df_validacao["timeout_total"].sum()),
      "| Pacotes perdidos:", int(df_validacao["pacotes_perdidos"].sum()))

sem_dados = df_validacao[df_validacao["pares"] < df_validacao["pares_solicitados"]]
if not sem_dados.empty:
    print("\nATENÇÃO: destinos com origens que não retornaram dados no período:")
    display(sem_dados[["destino_fqdn", "pares_solicitados", "pares"]])
if df_validacao["duplicatas"].sum() > 0:
    print("\nATENÇÃO: há registros duplicados (mesmo prb_id e timestamp).")

,regiao_destino,destino_fqdn,interval_s,pares_solicitados,pares,registros,registros_min_por_par,registros_max_por_par,duplicatas,timeout_total,pacotes_perdidos
0,América do Sul,br-for-as37468.anchors.atlas.ripe.net,240,5,5,12506,2476,2509,0,36,129
1,América do Sul,br-pgz-as53046.anchors.atlas.ripe.net,240,5,5,12497,2476,2515,0,27,99
2,América do Sul,br-poa-as19200.anchors.atlas.ripe.net,240,5,5,12540,2503,2513,0,3,15
3,América do Sul,br-sao-as12008.anchors.atlas.ripe.net,240,5,5,11628,1577,2513,0,470,1416
4,América do Sul,br-fec-as264479.anchors.atlas.ripe.net,240,5,5,12549,2499,2515,0,41,142
5,América do Norte,us-lax-as30058.anchors.atlas.ripe.net,240,5,5,12535,2484,2517,0,60,499
6,América do Norte,us-mes-as401332.anchors.atlas.ripe.net,240,5,5,12532,2482,2515,0,34,145
7,América do Norte,us-mia-as199610.anchors.atlas.ripe.net,240,5,5,12522,2482,2512,0,37,122
8,Ásia,jp-tyo-as63834.anchors.atlas.ripe.net,240,5,5,12527,2481,2518,0,3,71
9,Ásia,jp-tch-as63806.anchors.atlas.ripe.net,240,5,5,12554,2502,2514,0,15,212


Destinos: 20 | Pares: 100 | Registros: 249784
Duplicatas: 0 | Timeouts totais: 13348 | Pacotes perdidos: 40917


## 7. Dataset final (Contrato de Dados) e CSV

**Classificação relativa por par.** O RTT considerado normal depende da distância entre origem e destino (algo em torno de 220 ms é esperado num par transatlântico, mas seria grave dentro de um mesmo país). Por isso, o `status_real` é calculado em relação à linha de base de **cada par** (`par_id`), usando só registros sem perda de pacotes para estimar a média e o desvio-padrão da latência:

- **FALHA**: qualquer perda de pacotes, ou latência acima de média + 3 desvios-padrão;
- **RISCO**: latência entre média + 1 e média + 3 desvios-padrão;
- **OK**: latência abaixo de média + 1 desvio-padrão.

Colunas do CSV: `timestamp`, `par_id`, `destino_fqdn`, `regiao_destino`, `prb_id`, `pais_origem`, `ip`, `latencia_ms`, `perda_pacotes_pct`, `jitter_ms`, `status_real`.

In [ ]:
def classificar_status_relativo(df):
    sem_perda = df[df["perda_pacotes_pct"] == 0]
    estatisticas = sem_perda.groupby("par_id")["latencia_ms"].agg(latencia_media="mean", latencia_desvio="std")
    df = df.merge(estatisticas, on="par_id", how="left")

    limite_risco = df["latencia_media"] + df["latencia_desvio"]
    limite_falha = df["latencia_media"] + 3 * df["latencia_desvio"]

    df["status_real"] = np.select(
        [df["perda_pacotes_pct"] > 0, df["latencia_ms"] >= limite_falha, df["latencia_ms"] >= limite_risco],
        ["FALHA", "FALHA", "RISCO"],
        default="OK",
    )
    return df.drop(columns=["latencia_media", "latencia_desvio"])

In [ ]:
df_contrato = pd.concat(partes_contrato, ignore_index=True)
df_contrato = classificar_status_relativo(df_contrato)

# Conferência: toda origem está na mesma região do seu destino
regiao_origem = df_contrato["pais_origem"].map(REGIAO_POR_PAIS)
assert (regiao_origem == df_contrato["regiao_destino"]).all(), "Há origem fora da região do destino."

colunas = ["timestamp", "par_id", "destino_fqdn", "regiao_destino", "prb_id", "pais_origem", "ip",
           "latencia_ms", "perda_pacotes_pct", "jitter_ms", "status_real"]
df_final = df_contrato[colunas].sort_values(["destino_fqdn", "par_id", "timestamp"]).reset_index(drop=True)

caminho_csv_final = PASTA_RAW / "dataset_pares_multidestino.csv"
df_final.to_csv(caminho_csv_final, index=False, encoding="utf-8")
print("CSV final salvo em:", caminho_csv_final)
print("Linhas:", len(df_final), "| Pares:", df_final["par_id"].nunique(),
      "| Destinos:", df_final["destino_fqdn"].nunique(),
      "| Países de origem:", df_final["pais_origem"].nunique())

resumo = (df_final.groupby(["regiao_destino", "destino_fqdn"])
          .agg(pares=("par_id", "nunique"), paises_origem=("pais_origem", "nunique"), registros=("par_id", "size"))
          .reset_index())
display(resumo)

print("\nDistribuição de status_real:")
print(pd.concat([df_final["status_real"].value_counts().rename("registros"),
                 (df_final["status_real"].value_counts(normalize=True) * 100).round(2).rename("%")], axis=1))
display(df_final.head())

CSV final salvo em: /content/drive/MyDrive/ripe_atlas/raw/dataset_pares_multidestino.csv
Linhas: 249784 | Pares: 100 | Destinos: 20 | Países de origem: 46


,regiao_destino,destino_fqdn,pares,paises_origem,registros
0,América do Norte,us-lax-as30058.anchors.atlas.ripe.net,5,5,12535
1,América do Norte,us-mes-as401332.anchors.atlas.ripe.net,5,5,12532
2,América do Norte,us-mia-as199610.anchors.atlas.ripe.net,5,5,12522
3,América do Sul,br-fec-as264479.anchors.atlas.ripe.net,5,5,12549
4,América do Sul,br-for-as37468.anchors.atlas.ripe.net,5,5,12506
5,América do Sul,br-pgz-as53046.anchors.atlas.ripe.net,5,5,12497
6,América do Sul,br-poa-as19200.anchors.atlas.ripe.net,5,5,12540
7,América do Sul,br-sao-as12008.anchors.atlas.ripe.net,5,5,11628
8,América do Sul,co-bog-as273103.anchors.atlas.ripe.net,5,5,12546
9,América do Sul,co-ptx-as269907.anchors.atlas.ripe.net,5,5,12510



Distribuição de status_real:
             registros      %
status_real                  
OK              219586  87.91
FALHA            16619   6.65
RISCO            13579   5.44


,timestamp,par_id,destino_fqdn,regiao_destino,prb_id,pais_origem,ip,latencia_ms,perda_pacotes_pct,jitter_ms,status_real
0,2026-10-03 22:09:37+00:00,6488->br-fec-as264479.anchors.atlas.ripe.net,br-fec-as264479.anchors.atlas.ripe.net,América do Sul,6488,EC,143.255.248.149,159.28,0.0,0.11,OK
1,2026-10-03 22:13:40+00:00,6488->br-fec-as264479.anchors.atlas.ripe.net,br-fec-as264479.anchors.atlas.ripe.net,América do Sul,6488,EC,143.255.248.149,159.43,0.0,0.08,OK
2,2026-10-03 22:17:39+00:00,6488->br-fec-as264479.anchors.atlas.ripe.net,br-fec-as264479.anchors.atlas.ripe.net,América do Sul,6488,EC,143.255.248.149,159.55,0.0,0.18,OK
3,2026-10-03 22:21:49+00:00,6488->br-fec-as264479.anchors.atlas.ripe.net,br-fec-as264479.anchors.atlas.ripe.net,América do Sul,6488,EC,143.255.248.149,-1.00,100.0,0.00,FALHA
4,2026-10-03 22:25:45+00:00,6488->br-fec-as264479.anchors.atlas.ripe.net,br-fec-as264479.anchors.atlas.ripe.net,América do Sul,6488,EC,143.255.248.149,185.54,0.0,0.15,RISCO


## 8. Limitações

- **Origens amostradas:** de 3 a 5 por destino, sorteadas entre as Anchors da mesma região que participam da medição; não representam todos os caminhos possíveis.
- **Regiões definidas manualmente** em `REGIAO_POR_PAIS`; origens de países ausentes do mapa são ignoradas (a lista aparece na Seção 4).
- **Rótulo relativo:** `status_real` mede o desvio em relação ao comportamento normal do próprio par na janela coletada, não uma qualidade absoluta de rede. Se houver mudança de rota no período, a linha de base mistura os dois regimes.
- **`latencia_ms = -1`** é o valor que a API devolve quando todos os pacotes se perdem; nesses registros `perda_pacotes_pct = 100` e o status é `FALHA`.
- **Origens são Anchors** (infraestrutura estável, como datacenters e universidades), não redes domésticas.
- **Dados públicos:** medições podem ser interrompidas ou alteradas pelos responsáveis; `pares_selecionados.json` e os metadados em `raw/por_destino/` registram o que foi usado em cada coleta.